In [ ]:
import pandas as pd
fp = "../data/olist_prepared/SP_geo_revenue_by_city_2017.parquet"
df = pd.read_parquet(fp)
geo_cols = df.columns.tolist()

In [ ]:
cities = df.index.tolist()

In [ ]:
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import euclidean_distances

In [ ]:
X_dist = euclidean_distances(df.values, df.values)

In [ ]:
import numpy as np
import numpy.linalg as la

In [ ]:
start_number = 10
end_number = 1000
num_values = 10

# Generate 10 evenly spaced values between start_number and end_number (inclusive)
window_sizes = np.linspace(start_number, end_number, num_values)

In [ ]:
# Construct similarity matrix using a Gaussian kernel
num_conn_comp = {}
EPS = 0.01
for w in window_sizes:
    sigma = w # Adjust sigma for desired width of similarity
    similarity_matrix = np.exp(- X_dist**2 / (2 * sigma**2))
    degree_matrix = np.diag(np.sum(similarity_matrix, axis=1))
    laplacian_matrix = degree_matrix - similarity_matrix
    eigvals, eigvecs = la.eig(laplacian_matrix)
    sorted_indices = np.argsort(eigvals)
    # Sort eigenvalues
    sorted_eigvals = eigvals[sorted_indices]
    
    # Sort eigenvectors by applying the same indices to the columns
    sorted_eigvecs = eigvecs[:, sorted_indices]

    sorted_eigvals[np.abs(sorted_eigvals) < EPS] = 0
    num_zero_eig_vals = np.sum(sorted_eigvals == 0)
    num_conn_comp[w] = num_zero_eig_vals

In [ ]:
df_sexp = pd.DataFrame.from_dict(num_conn_comp, orient="index").reset_index()
df_sexp.columns = ["sigma", "num_conn_comp"]

In [ ]:
df_sexp